# Official CODI parameter-aware endpoint spectral experiment

This experiment selects final-two-block residual PCs by their induced LoRA-parameter-gradient cosine with the numeric-answer gradient. It uses fresh seed-41 partitions, deterministic Hutchinson norm sketches, held-out equal-update-norm controls, and a checksummed resumable export. Enable **Internet** and a **T4 or newer GPU**, then use **Save Version → Save & Run All**.

## 1. Configuration

In [ ]:
REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
RUN_COMMIT = "main"  # Replace with the immutable commit after pushing.
REPO_DIR = "/kaggle/working/latent-reasoning"

REPRODUCTION_SUMMARY_INPUT = ""
RESUME_INPUT = ""  # Optional prior parameter-aware export root.
RUN_REPRODUCTION_GATE_IF_MISSING = True
RUN_SMOKE = True
RUN_FULL_COLLECTION = True
RUN_UTILITY = True

RESIDUAL_FIT_EXAMPLES = 1024
DIRECTION_SELECTION_EXAMPLES = 1024
UPDATE_EXAMPLES = 256
VALIDATION_EXAMPLES = 256
FIT_BATCH_SIZE = 16
SELECTION_BATCH_SIZE = 8
UTILITY_BATCH_SIZE = 4
SAMPLING_SEED = 41
RANDOM_BASIS_SEED = 20260805
PROBE_SEED = 314159
CANDIDATE_STATES = [11, 12]
CANDIDATE_PC_COUNT = 64
HUTCHINSON_PROBES = 8
MINIMUM_SPLIT_Z = 1.645
SELECTION_FDR = 0.05
MAXIMUM_RANK_PER_STATE = 8
RELATIVE_UPDATE_NORM = 1e-4
PRECISION = "float32"
BOOTSTRAP_SAMPLES = 10000
BOOTSTRAP_SEED = 0

UPLOAD_AS_KAGGLE_DATASET = False
KAGGLE_DATASET_HANDLE = "jonraza15/official-codi-endpoint-parameter-aware"

## 2. Install and pin the repository

In [ ]:
import datetime, hashlib, json, os, pathlib, shutil, subprocess, sys

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("Hugging Face authentication: Kaggle secret loaded")
except Exception:
    print("Hugging Face authentication: public access")

repo = pathlib.Path(REPO_DIR)
if not (repo / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
target = f"origin/{RUN_COMMIT}" if RUN_COMMIT == "main" else RUN_COMMIT
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--detach", target], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements-official-codi.txt")], check=True)
os.chdir(REPO_DIR)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Checked out:", commit)
if RUN_COMMIT == "main":
    print("PIN RUN_COMMIT BEFORE THE FINAL SAVE VERSION RUN:", commit)

## 3. Hardware and implementation tests

In [ ]:
import torch, transformers
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator"
capability = torch.cuda.get_device_capability(0)
print("Torch:", torch.__version__, "Transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0), "capability:", capability)
assert capability >= (7, 0), "Use a T4 or newer GPU"
subprocess.run([
    sys.executable, "-m", "pytest", "-q",
    "tests/test_official_codi.py",
    "tests/test_official_codi_target_utility.py",
    "tests/test_endpoint_answer_conditioned.py",
    "tests/test_endpoint_parameter_aware.py",
    "tests/test_official_codi_endpoint_parameter_aware_analysis.py",
], cwd=REPO_DIR, check=True)

## 4. Durable paths, logs, and optional resume

In [ ]:
OUTPUT_ROOT = repo / "outputs" / "official_codi_endpoint_parameter_aware"
REPORT_ROOT = repo / "reports" / "official_codi_endpoint_parameter_aware"
LOG_ROOT = repo / "logs" / "official_codi_endpoint_parameter_aware"
VALIDATION_ROOT = repo / "outputs" / "official_codi_gpt2"
for path in (OUTPUT_ROOT, REPORT_ROOT, LOG_ROOT, VALIDATION_ROOT):
    path.mkdir(parents=True, exist_ok=True)

def run_persisted(command, log_name):
    log_path = LOG_ROOT / log_name
    print("Starting:", " ".join(map(str, command)), flush=True)
    print("Persistent log:", log_path, flush=True)
    with log_path.open("a", encoding="utf-8", buffering=1) as log:
        stamp = datetime.datetime.now(datetime.timezone.utc).isoformat()
        log.write(f"\n=== {stamp} {' '.join(map(str, command))} ===\n")
        process = subprocess.Popen(command, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        code = process.wait()
    if code != 0:
        raise RuntimeError(f"Command failed with exit code {code}; inspect {log_path}")
    return log_path

if RESUME_INPUT:
    resume_root = pathlib.Path(RESUME_INPUT)
    manifests = list(resume_root.rglob("official_codi_endpoint_parameter_aware/collection_seed41/run_manifest.json"))
    assert manifests, "No parameter-aware collection manifest found"
    request_hashes = {json.loads(path.read_text()).get("request_sha256") for path in manifests}
    assert len(request_hashes) == 1, f"Incompatible resume trees: {manifests}"
    source = sorted(manifests, key=lambda p: (len(p.parts), p.as_posix()))[0].parents[1]
    shutil.copytree(source, OUTPUT_ROOT, dirs_exist_ok=True)
    print("Restored outputs from:", source)
else:
    print("Starting without prior parameter-aware outputs")

## 5. Locate or run the official full-GSM8K reproduction gate

In [ ]:
EXPECTED_REVISION = "fd641b3d3edc59e4f534b55588e906588c9e36bb"
def passed_summary(path):
    try:
        payload = json.loads(path.read_text())
    except Exception:
        return False
    gate = payload.get("accuracy_gate", payload.get("gate"))
    status = gate.get("status") if isinstance(gate, dict) else gate
    count = payload.get("evaluated_counts", {}).get("gsm8k")
    revision = payload.get("checkpoint_revision")
    return status == "passed" and count == 1319 and revision in {None, EXPECTED_REVISION}

if REPRODUCTION_SUMMARY_INPUT:
    REPRODUCTION_SUMMARY = pathlib.Path(REPRODUCTION_SUMMARY_INPUT)
    assert passed_summary(REPRODUCTION_SUMMARY)
else:
    candidates = [p for p in pathlib.Path("/kaggle/input").rglob("summary.json") if passed_summary(p)]
    candidates += [p for p in VALIDATION_ROOT.rglob("summary.json") if passed_summary(p)]
    if not candidates:
        assert RUN_REPRODUCTION_GATE_IF_MISSING
        run_persisted([sys.executable, "-u", "-m", "src.eval.official_codi", "--config", "configs/official_codi_gpt2.yaml", "--datasets", "gsm8k", "--limit", "0", "--device", "cuda", "--output-dir", str(VALIDATION_ROOT)], "official_codi_gsm8k_gate.log")
        candidates = [p for p in VALIDATION_ROOT.rglob("summary.json") if passed_summary(p)]
    assert candidates, "A passed full-GSM8K summary is required"
    REPRODUCTION_SUMMARY = sorted(candidates, key=lambda p: p.as_posix())[0]
print("Reproduction summary:", REPRODUCTION_SUMMARY)

## 6. Commands and mandatory double-backward smoke path

In [ ]:
def collection_command(root, fit_n, select_n, update_n, validation_n, fit_batch, select_batch, pc_count, probes, min_z, fdr, max_rank, save_every):
    return [sys.executable, "-u", "scripts/collect_official_codi_endpoint_parameter_aware.py", "--config", "configs/official_codi_gpt2.yaml", "--reproduction-summary", str(REPRODUCTION_SUMMARY), "--output-dir", str(root), "--residual-fit-examples", str(fit_n), "--direction-selection-examples", str(select_n), "--update-examples", str(update_n), "--validation-examples", str(validation_n), "--fit-batch-size", str(fit_batch), "--selection-batch-size", str(select_batch), "--save-every", str(save_every), "--sampling-seed", str(SAMPLING_SEED), "--random-basis-seed", str(RANDOM_BASIS_SEED), "--probe-seed", str(PROBE_SEED), "--candidate-states", *map(str, CANDIDATE_STATES), "--candidate-pc-count", str(pc_count), "--hutchinson-probes", str(probes), "--minimum-split-z", str(min_z), "--selection-fdr", str(fdr), "--maximum-rank-per-state", str(max_rank), "--parity-examples", "4", "--precision", PRECISION, "--device", "cuda"]

def utility_command(root, basis, bootstrap_samples=BOOTSTRAP_SAMPLES):
    return [sys.executable, "-u", "scripts/run_official_codi_endpoint_parameter_aware_utility.py", "--config", "configs/official_codi_gpt2.yaml", "--reproduction-summary", str(REPRODUCTION_SUMMARY), "--basis", str(basis), "--output-dir", str(root), "--batch-size", str(UTILITY_BATCH_SIZE), "--seed", str(SAMPLING_SEED), "--relative-update-norm", str(RELATIVE_UPDATE_NORM), "--bootstrap-samples", str(bootstrap_samples), "--bootstrap-seed", str(BOOTSTRAP_SEED), "--precision", PRECISION, "--device", "cuda"]

SMOKE_COLLECTION = OUTPUT_ROOT / "smoke_seed41" / "collection"
SMOKE_UTILITY = OUTPUT_ROOT / "smoke_seed41" / "utility"
if RUN_SMOKE:
    run_persisted(collection_command(SMOKE_COLLECTION, 16, 64, 8, 8, 8, 4, 64, 2, -100.0, 1.0, 2, 8), "smoke_parameter_aware_collection.log")
    smoke_basis = torch.load(SMOKE_COLLECTION / "basis.pt", map_location="cpu", weights_only=False)
    assert smoke_basis["metadata"]["native_parity_gate"]["status"] == "passed"
    assert smoke_basis["selection"]["status"] == "candidate_selected", smoke_basis["selection"]
    run_persisted(utility_command(SMOKE_UTILITY, SMOKE_COLLECTION / "basis.pt", 500), "smoke_parameter_aware_utility.log")
    assert json.loads((SMOKE_UTILITY / "run_manifest.json").read_text())["state"] == "complete"
    print("Parameter-aware smoke path complete")
else:
    print("Smoke skipped; run only with an equivalent tested commit")

## 7. Fit residual PCs and select parameter-aware directions

In [ ]:
COLLECTION_ROOT = OUTPUT_ROOT / "collection_seed41"
BASIS_PATH = COLLECTION_ROOT / "basis.pt"
if RUN_FULL_COLLECTION:
    run_persisted(collection_command(COLLECTION_ROOT, RESIDUAL_FIT_EXAMPLES, DIRECTION_SELECTION_EXAMPLES, UPDATE_EXAMPLES, VALIDATION_EXAMPLES, FIT_BATCH_SIZE, SELECTION_BATCH_SIZE, CANDIDATE_PC_COUNT, HUTCHINSON_PROBES, MINIMUM_SPLIT_Z, SELECTION_FDR, MAXIMUM_RANK_PER_STATE, 128), "collection_parameter_aware_seed41.log")
assert BASIS_PATH.is_file(), f"Missing basis: {BASIS_PATH}"
basis_payload = torch.load(BASIS_PATH, map_location="cpu", weights_only=False)
manifest = json.loads((COLLECTION_ROOT / "run_manifest.json").read_text())
assert manifest["state"] == "complete"
assert manifest["native_parity_gate"]["status"] == "passed"
assert manifest["sampling"]["excluded_unique_questions"] == 8072
selection = basis_payload["selection"]
print("Selection status:", selection["status"])
print("Rank by state:", selection["rank_by_state"])
print("Selected PCs:", selection["selected_pc_indices_by_state"])
print("Basis SHA256:", manifest["basis_sha256"])

## 8. Run held-out utility only if selection produced a candidate

In [ ]:
UTILITY_ROOT = OUTPUT_ROOT / "utility_seed41"
candidate_selected = selection["status"] == "candidate_selected"
if candidate_selected and RUN_UTILITY:
    run_persisted(utility_command(UTILITY_ROOT, BASIS_PATH), "utility_parameter_aware_seed41.log")
if candidate_selected:
    utility_manifest = json.loads((UTILITY_ROOT / "run_manifest.json").read_text())
    assert utility_manifest["state"] == "complete"
    assert len(utility_manifest["completed_batches"]) == 64
    print("Held-out utility complete")
else:
    print("No split-stable positive parameter cosines; utility correctly skipped")

## 9. Build the final registered decision

In [ ]:
from IPython.display import Markdown, display
REPORT_PATH = REPORT_ROOT / "official_codi_endpoint_parameter_aware_seed41.json"
analysis_command = [sys.executable, "scripts/analyze_official_codi_endpoint_parameter_aware.py", "--basis", str(BASIS_PATH), "--output", str(REPORT_PATH)]
if candidate_selected:
    analysis_command.extend(["--utility", str(UTILITY_ROOT)])
run_persisted(analysis_command, "analyze_parameter_aware_seed41.log")
report = json.loads(REPORT_PATH.read_text())
display(Markdown(REPORT_PATH.with_suffix(".md").read_text()))
print("FINAL STATUS:", report["status"])
print("TRAINING AUTHORIZED:", report["training_authorized"])

## 10. Build a checksummed resumable export

In [ ]:
EXPORT_ROOT = pathlib.Path("/kaggle/working/official_codi_endpoint_parameter_aware_export")
if EXPORT_ROOT.exists():
    shutil.rmtree(EXPORT_ROOT)
export_repo = EXPORT_ROOT / "latent-reasoning"
shutil.copytree(OUTPUT_ROOT, export_repo / "outputs" / "official_codi_endpoint_parameter_aware")
shutil.copytree(REPORT_ROOT, export_repo / "reports" / "official_codi_endpoint_parameter_aware")
shutil.copytree(LOG_ROOT, export_repo / "logs" / "official_codi_endpoint_parameter_aware")
validation_export = export_repo / "outputs" / "official_codi_gpt2_reproduction"
validation_export.mkdir(parents=True, exist_ok=True)
shutil.copy2(REPRODUCTION_SUMMARY, validation_export / "summary.json")
(EXPORT_ROOT / "RUN_COMMIT.txt").write_text(commit + "\n")
(EXPORT_ROOT / "RUN_INSTRUCTIONS.txt").write_text("Attach this parameter-aware dataset and set RESUME_INPUT to its root. Earlier endpoint exports are not compatible resume inputs.\n")
files = sorted(path for path in EXPORT_ROOT.rglob("*") if path.is_file())
lines = [f"{hashlib.sha256(path.read_bytes()).hexdigest()}  {path.relative_to(EXPORT_ROOT).as_posix()}" for path in files]
(EXPORT_ROOT / "SHA256SUMS.txt").write_text("\n".join(lines) + "\n")
all_files = [path for path in EXPORT_ROOT.rglob("*") if path.is_file()]
print("Export root:", EXPORT_ROOT)
print("Files:", len(all_files), "Size MiB:", sum(path.stat().st_size for path in all_files) / 2**20)
print("Use Save Version with outputs enabled.")

## 11. Optional direct Kaggle Dataset upload

In [ ]:
if UPLOAD_AS_KAGGLE_DATASET:
    import kagglehub
    kagglehub.dataset_upload(KAGGLE_DATASET_HANDLE, str(EXPORT_ROOT), version_notes=f"Parameter-aware official CODI endpoint experiment at {commit}")
    print("Dataset upload complete:", KAGGLE_DATASET_HANDLE)
else:
    print("Direct upload skipped; Save Version with outputs enabled is sufficient.")

## Interpretation

If no PC survives both parameter-cosine selection halves, the experiment closes before utility. If a candidate is selected, only superiority to all five controls plus positive median held-out parameter-gradient cosine authorizes a separately preregistered training study.